# AI Grid Bot - Training on Google Colab (GPU)

1. Upload project files
2. Train LSTM models on GPU (fast!)
3. Download trained models

In [ ]:
# Step 1: Install dependencies
!pip install ccxt ta scikit-learn joblib -q

In [ ]:
# Step 2: Check GPU
import tensorflow as tf
print(f'TensorFlow: {tf.__version__}')
print(f'GPU: {tf.config.list_physical_devices("GPU")}')
print('GPU available!' if tf.config.list_physical_devices('GPU') else 'CPU only')

In [ ]:
# Step 3: Upload project files
# Option A: Upload from Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Copy project to Colab
# !cp -r /content/drive/MyDrive/ai-grid-bot /content/ai-grid-bot

# Option B: Clone from GitHub (if you have a repo)
# !git clone https://github.com/YOUR_USER/ai-grid-bot.git

# Option C: Upload files manually using the sidebar

In [ ]:
%%writefile /content/ai/data_collector.py
"""Data collector for historical OHLCV data."""
import pandas as pd
import os
import time

DATA_DIR = '/content/data'

def fetch_historical_data(exchange, symbol, timeframe='1h', days=90):
    os.makedirs(DATA_DIR, exist_ok=True)
    since = exchange.milliseconds() - days * 24 * 60 * 60 * 1000
    all_ohlcv = []
    while since < exchange.milliseconds():
        ohlcv = exchange.fetch_ohlcv(symbol, timeframe, since=since, limit=1000)
        if not ohlcv:
            break
        all_ohlcv.extend(ohlcv)
        since = ohlcv[-1][0] + 1
        time.sleep(exchange.rateLimit / 1000)
    df = pd.DataFrame(all_ohlcv, columns=['timestamp', 'open', 'high', 'low', 'close', 'volume'])
    df['timestamp'] = pd.to_datetime(df['timestamp'], unit='ms')
    df = df.drop_duplicates(subset=['timestamp']).reset_index(drop=True)
    return df

def save_to_csv(df, symbol, timeframe):
    os.makedirs(DATA_DIR, exist_ok=True)
    path = os.path.join(DATA_DIR, f"{symbol.replace('/', '_')}_{timeframe}.csv")
    df.to_csv(path, index=False)

def update_data(exchange, symbol, timeframe):
    return fetch_historical_data(exchange, symbol, timeframe, days=5)

In [ ]:
# Step 4: Create ai module files
!mkdir -p /content/ai /content/data /content/models
!touch /content/ai/__init__.py

# Upload your feature_engineer.py and model.py
# Or copy from Google Drive:
# !cp /content/drive/MyDrive/ai-grid-bot/ai/*.py /content/ai/

In [ ]:
%%writefile /content/ai/feature_engineer.py
"""Feature engineering for LSTM model."""
import numpy as np
import pandas as pd
import ta
from sklearn.preprocessing import MinMaxScaler

def add_indicators(df):
    df = df.copy()
    df['rsi'] = ta.momentum.RSIIndicator(df['close'], window=14).rsi()
    macd = ta.trend.MACD(df['close'], window_slow=26, window_fast=12, window_sign=9)
    df['macd'] = macd.macd()
    df['macd_signal'] = macd.macd_signal()
    df['macd_hist'] = macd.macd_diff()
    bb = ta.volatility.BollingerBands(df['close'], window=20, window_dev=2)
    df['bb_upper'] = bb.bollinger_hband()
    df['bb_lower'] = bb.bollinger_lband()
    df['bb_width'] = (df['bb_upper'] - df['bb_lower']) / df['close']
    df['bb_position'] = (df['close'] - df['bb_lower']) / (df['bb_upper'] - df['bb_lower'])
    df['atr'] = ta.volatility.AverageTrueRange(df['high'], df['low'], df['close'], window=14).average_true_range()
    df['atr_pct'] = df['atr'] / df['close']
    df['ema_20'] = ta.trend.EMAIndicator(df['close'], window=20).ema_indicator()
    df['ema_50'] = ta.trend.EMAIndicator(df['close'], window=50).ema_indicator()
    df['ema_cross'] = (df['ema_20'] - df['ema_50']) / df['close']
    stoch = ta.momentum.StochasticOscillator(df['high'], df['low'], df['close'], window=14, smooth_window=3)
    df['stoch_k'] = stoch.stoch()
    df['stoch_d'] = stoch.stoch_signal()
    df['volume_sma'] = df['volume'].rolling(window=20).mean()
    df['volume_ratio'] = df['volume'] / df['volume_sma'].replace(0, 1)
    df['returns_1'] = df['close'].pct_change(1)
    df['returns_6'] = df['close'].pct_change(6)
    df['returns_24'] = df['close'].pct_change(24)
    return df

FEATURE_COLUMNS = [
    'rsi', 'macd', 'macd_signal', 'macd_hist',
    'bb_width', 'bb_position', 'atr_pct',
    'ema_cross', 'stoch_k', 'stoch_d',
    'volume_ratio', 'returns_1', 'returns_6', 'returns_24',
]

def create_target(df, horizon=6, threshold=0.005):
    future_return = df['close'].shift(-horizon) / df['close'] - 1
    target = pd.Series(1, index=df.index)
    target[future_return > threshold] = 2
    target[future_return < -threshold] = 0
    return target

def create_sequences(features, targets, seq_length=48):
    X, y = [], []
    for i in range(seq_length, len(features) - 1):
        X.append(features[i - seq_length:i])
        y.append(targets[i])
    return np.array(X), np.array(y)

def prepare_training_data(df, seq_length=48, horizon=6):
    df = add_indicators(df)
    df['target'] = create_target(df, horizon=horizon)
    df = df.dropna(subset=FEATURE_COLUMNS + ['target']).reset_index(drop=True)
    features = df[FEATURE_COLUMNS].values
    targets = df['target'].values.astype(int)
    scaler = MinMaxScaler()
    split = int(len(features) * 0.8)
    scaler.fit(features[:split])
    features_scaled = scaler.transform(features)
    X, y = create_sequences(features_scaled, targets, seq_length)
    split_idx = int(len(X) * 0.8)
    return X[:split_idx], y[:split_idx], X[split_idx:], y[split_idx:], scaler

In [ ]:
%%writefile /content/ai/model.py
"""LSTM model."""
import os
import numpy as np
import joblib

MODEL_DIR = '/content/models'

def build_lstm_model(seq_length, n_features, n_classes=3):
    from tensorflow import keras
    model = keras.Sequential([
        keras.layers.LSTM(64, return_sequences=True, input_shape=(seq_length, n_features)),
        keras.layers.Dropout(0.2),
        keras.layers.LSTM(32, return_sequences=False),
        keras.layers.Dropout(0.2),
        keras.layers.Dense(16, activation='relu'),
        keras.layers.Dense(n_classes, activation='softmax'),
    ])
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.001),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy'],
    )
    return model

def train_model(model, X_train, y_train, X_val, y_val, epochs=50, batch_size=32):
    from tensorflow import keras
    callbacks = [
        keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=10, restore_best_weights=True),
        keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5),
    ]
    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=epochs, batch_size=batch_size,
        callbacks=callbacks, verbose=1,
    )
    return history

def save_model(model, scaler, symbol):
    os.makedirs(MODEL_DIR, exist_ok=True)
    safe = symbol.replace('/', '_')
    model.save(os.path.join(MODEL_DIR, f'{safe}_lstm.keras'))
    joblib.dump(scaler, os.path.join(MODEL_DIR, f'{safe}_scaler.pkl'))

def load_model(symbol):
    from tensorflow import keras
    safe = symbol.replace('/', '_')
    model_path = os.path.join(MODEL_DIR, f'{safe}_lstm.keras')
    scaler_path = os.path.join(MODEL_DIR, f'{safe}_scaler.pkl')
    if not os.path.exists(model_path):
        return None, None
    return keras.models.load_model(model_path), joblib.load(scaler_path)

In [ ]:
# Step 5: TRAIN ALL MODELS (GPU - should be fast!)
import sys
sys.path.insert(0, '/content')

import ccxt
from ai.data_collector import fetch_historical_data, save_to_csv
from ai.feature_engineer import prepare_training_data, FEATURE_COLUMNS
from ai.model import build_lstm_model, train_model, save_model

SYMBOLS = ['BTC/USDT', 'ETH/USDT', 'SOL/USDT', 'BNB/USDT', 'DOGE/USDT', 'ADA/USDT', 'LINK/USDT']
SEQ_LENGTH = 48
HORIZON = 6
EPOCHS = 50

exchange = ccxt.binance({'enableRateLimit': True})

results = {}
for symbol in SYMBOLS:
    print(f'\n{"="*60}')
    print(f'  Training: {symbol}')
    print(f'{"="*60}')
    
    df = fetch_historical_data(exchange, symbol, '1h', days=90)
    save_to_csv(df, symbol, '1h')
    print(f'  Data: {len(df)} candles')
    
    if len(df) < 200:
        print(f'  SKIPPED: not enough data')
        continue
    
    X_train, y_train, X_val, y_val, scaler = prepare_training_data(df, SEQ_LENGTH, HORIZON)
    print(f'  Train: {len(X_train)}, Val: {len(X_val)}')
    
    model = build_lstm_model(SEQ_LENGTH, len(FEATURE_COLUMNS), n_classes=3)
    history = train_model(model, X_train, y_train, X_val, y_val, epochs=EPOCHS)
    
    val_acc = max(history.history['val_accuracy'])
    results[symbol] = val_acc
    print(f'  Best val_accuracy: {val_acc:.4f}')
    
    save_model(model, scaler, symbol)
    print(f'  Model saved!')

print(f'\n{"="*60}')
print('  ALL MODELS TRAINED!')
for sym, acc in results.items():
    print(f'  {sym}: {acc:.4f}')
print(f'{"="*60}')

In [ ]:
# Step 6: Download models
import shutil

# Pack models into zip
shutil.make_archive('/content/models_trained', 'zip', '/content/models')

# Download
from google.colab import files
files.download('/content/models_trained.zip')

print('Download models_trained.zip and extract to ai-grid-bot/models/')